# Tutorial 02: Agent Routing from Discovered Capabilities (Offline-First)

A focused, implementation-driven walkthrough of routing logic in `A2A/02_ Agent Routing`, designed to run correctly even when no local agent servers are running.

## 1) Where We Are in the Journey

In `01_capability_discovery`, the system learned to build a live registry from agent metadata and tool schemas.
That gave us visibility into what each agent can do, but not yet a decision rule for selecting an agent per query.
This tutorial exists to implement that decision layer: routing from user intent to the most suitable discovered agent.

## 2) What You Will Learn

- How to keep capability discovery and routing as separate but connected concerns
- How the `route(query, registry)` function uses tool names/descriptions for matching
- Why normalized query/tool text is a practical baseline router strategy
- What trade-offs exist in keyword routing before adding LLM-based reasoning
- How this step prepares clean handoff to invocation in the next stage

## 3) Why This Matters

Discovery tells you what is available; routing decides what to do now.
In A2A systems, routing quality directly impacts correctness, latency, and user trust.
This folder demonstrates a lightweight, transparent routing strategy that is easy to debug and iterate before moving to more advanced policies.

## 4) Core Concept Deep Dive

### Intuition

Routing is a matching problem between user intent and discovered capabilities. The controller does not execute business logic yet; it first decides ownership of the request.

### System design thinking

This step composes two layers:

1. `discover_capabilities()` builds the runtime capability registry from live agents.
2. `route(query, registry)` chooses an agent by comparing query text with each tool's name and description.

This keeps the architecture modular: discovery can evolve independently from route policy.

### Trade-offs

- Strengths: deterministic behavior, low cost, straightforward debugging
- Weaknesses: brittle language matching, sensitivity to wording, limited semantic understanding
- Design implication: keyword routing is a baseline policy, not the end-state for complex user intent

### Offline-first teaching choice

Because servers are currently unavailable, this tutorial provides a faithful mock registry with the same schema as real `/agent-info` + `/tools` responses. That preserves routing behavior and learning outcomes without redesigning target-folder logic.

## 5) Code Walkthrough (Only `A2A/02_ Agent Routing`)

Primary file:

- `routing.ipynb`: discovery + routing flow for sample math/finance queries

Key code elements:

- `AGENTS`: target base URLs for discovery
- `discover_capabilities()`: merges `/agent-info` and `/tools` into a unified registry
- `route(query, registry)`: lowercases text and matches query against tool name/description tokens
- demo cell: routes `calculate interest...` and `add ...` queries

Dependency on reusable foundations:

This folder relies on `00_Agents`-style contracts (`/agent-info`, `/tools`) to keep registry and routing logic generic.

Offline tutorial adaptation:

To keep the tutorial executable with no running servers, we keep the same registry shape and routing code while sourcing registry data from a local mock fallback when health checks fail.

In [5]:
# 6) Executable Cell: Setup and imports
import requests

AGENTS = [
    "http://localhost:8000",  # math-agent
    "http://localhost:8001"   # finance-agent
]

# Mock registry source shaped exactly like discovery output.
MOCK_REGISTRY = [
    {
        "agent": "math-agent",
        "description": "Handles mathematical operations like addition",
        "endpoint": "http://localhost:8000/invoke",
        "tools": [
            {
                "name": "add",
                "description": "Add two numbers",
                "input_schema": {
                    "type": "object",
                    "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
                    "required": ["a", "b"]
                }
            }
        ]
    },
    {
        "agent": "finance-agent",
        "description": "Handles financial calculations like simple interest",
        "endpoint": "http://localhost:8001/invoke",
        "tools": [
            {
                "name": "calculate_interest",
                "description": "Calculate simple interest",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "principal": {"type": "number"},
                        "rate": {"type": "number"},
                        "time": {"type": "number"}
                    },
                    "required": ["principal", "rate", "time"]
                }
            }
        ]
    }
]

print('Configured routing targets:', AGENTS)
print('Offline fallback registry entries:', len(MOCK_REGISTRY))

Configured routing targets: ['http://localhost:8000', 'http://localhost:8001']
Offline fallback registry entries: 2


In [6]:
# 7) Executable Cell: Connectivity pre-check
def check_agents(agents):
    status = {}
    for url in agents:
        try:
            resp = requests.get(f"{url}/agent-info", timeout=2)
            if resp.status_code == 200:
                status[url] = 'UP (200)'
            else:
                status[url] = f'DOWN (HTTP {resp.status_code})'
        except Exception as e:
            status[url] = f'DOWN ({type(e).__name__})'
    return status

health = check_agents(AGENTS)
health

{'http://localhost:8000': 'DOWN (HTTP 404)',
 'http://localhost:8001': 'DOWN (ConnectionError)'}

In [7]:
# 8) Executable Cell: Capability discovery with offline fallback
from pprint import pprint

def discover_capabilities_online():
    registry = []

    for url in AGENTS:
        info = requests.get(f"{url}/agent-info", timeout=2).json()
        tools = requests.get(f"{url}/tools", timeout=2).json()

        registry.append({
            "agent": info["name"],
            "description": info["description"],
            "endpoint": info["endpoint"],
            "tools": tools["tools"]
        })

    return registry

def discover_capabilities_offline():
    return MOCK_REGISTRY

def get_registry(prefer_online=True):
    all_up = all(v.startswith('UP') for v in health.values())
    if prefer_online and all_up:
        return discover_capabilities_online(), 'online'
    return discover_capabilities_offline(), 'offline'

registry, mode = get_registry(prefer_online=True)
print('Registry source mode:', mode)
pprint(registry)

Registry source mode: offline
[{'agent': 'math-agent',
  'description': 'Handles mathematical operations like addition',
  'endpoint': 'http://localhost:8000/invoke',
  'tools': [{'description': 'Add two numbers',
             'input_schema': {'properties': {'a': {'type': 'number'},
                                             'b': {'type': 'number'}},
                              'required': ['a', 'b'],
                              'type': 'object'},
             'name': 'add'}]},
 {'agent': 'finance-agent',
  'description': 'Handles financial calculations like simple interest',
  'endpoint': 'http://localhost:8001/invoke',
  'tools': [{'description': 'Calculate simple interest',
             'input_schema': {'properties': {'principal': {'type': 'number'},
                                             'rate': {'type': 'number'},
                                             'time': {'type': 'number'}},
                              'required': ['principal', 'rate', 'time'],
          

In [8]:
# 9) Executable Cell: Routing logic from target notebook
# Kept intentionally aligned with `routing.ipynb` behavior.
def route(query, registry):
    query = query.lower()

    for agent in registry:
        for tool in agent["tools"]:
            tool_name = tool["name"].lower()
            tool_desc = tool["description"].lower()

            if tool_name in query or any(word in query for word in tool_desc.split()):
                return agent["agent"]

    return "No suitable agent found"

In [9]:
# 10) Executable Cell: Route sample queries (works offline and online)
registry, mode = get_registry(prefer_online=True)

query1 = "calculate interest for 1000"
query2 = "add 5 and 4"
query3 = "book a hotel"

print('Registry mode used:', mode)
print('Query:', query1)
print('Agent:', route(query1, registry))

print('\nQuery:', query2)
print('Agent:', route(query2, registry))

print('\nQuery:', query3)
print('Agent:', route(query3, registry))

Registry mode used: offline
Query: calculate interest for 1000
Agent: finance-agent

Query: add 5 and 4
Agent: math-agent

Query: book a hotel
Agent: No suitable agent found


In [10]:
# 11) Executable Cell: Focused routing behavior tests (offline)
test_cases = [
    ("calculate interest for 2500", "finance-agent"),
    ("please add 12 and 7", "math-agent"),
    ("summarize this article", "No suitable agent found")
]

passed = 0
for query, expected in test_cases:
    predicted = route(query, MOCK_REGISTRY)
    ok = predicted == expected
    passed += int(ok)
    print(f"query={query!r} | predicted={predicted!r} | expected={expected!r} | pass={ok}")

print(f"\nRouting tests passed: {passed}/{len(test_cases)}")

query='calculate interest for 2500' | predicted='finance-agent' | expected='finance-agent' | pass=True
query='please add 12 and 7' | predicted='math-agent' | expected='math-agent' | pass=True
query='summarize this article' | predicted='No suitable agent found' | expected='No suitable agent found' | pass=True

Routing tests passed: 3/3


## 6) System Flow Explanation

1. The controller reads `AGENTS` base URLs.
2. For each URL, it discovers metadata (`/agent-info`) and tool schema (`/tools`).
3. It builds a registry entry with agent identity, endpoint, and tools.
4. At query time, router lowercases the query and compares against each tool name/description.
5. On first match, router returns the selected agent name.
6. If no match exists, it returns a no-agent-found fallback response.

## 7) Important Concepts You Might Miss

- Routing here is capability-based, not hardcoded if/else by agent identity.
- Description token matching can over-trigger on common words if tool descriptions are vague.
- Returning only agent name is enough for this step, while next-step invocation usually needs the full agent object.
- Offline fallback is a teaching strategy; the data shape must match real discovery output to keep conclusions valid.
- Discovery freshness still matters in production, where stale online registries can misroute requests.

## 8) Common Mistakes / Pitfalls

- Assuming tutorials must fail if servers are down, instead of using schema-faithful offline fixtures.
- Port mismatch between running services and `AGENTS` list when switching to online mode.
- Assuming keyword overlap always implies correct intent mapping.
- Ignoring ambiguous queries where multiple tools may partially match.
- Forgetting fallback handling when no suitable agent is found.

## 9) Key Takeaways

- `A2A/02_ Agent Routing` turns discovered capabilities into routing decisions.
- The route function is intentionally simple, transparent, and deterministic.
- You can validate routing behavior offline if your mock registry mirrors production schema.
- Capability schema quality directly affects routing quality.
- This stage completes selection and prepares clean handoff to invocation.

## 10) What’s Next (Strict Preview)

Next, `03_agent_invocation` addresses execution after selection: building payloads and calling the chosen agent endpoint.
It solves the gap between selecting an agent and actually producing results from that agent.
This matters because routing without invocation is only a plan, not an outcome.
You will move from decision logic to end-to-end query execution.